# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedman47/ML-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I already compared a hand-written rule (stale × visible) with a learned model in the Week 1–2 notebooks,
and Lane 2 extends that into a ranked review queue with reason codes. A review team has limited capacity,
so what matters is how good the top of the list is (Precision@K),
not overall accuracy. I may change lanes before the end of Week 4.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Decision: which already-published pages a content team should review first because their search visibility looks like it is declining.

Who acts: a content reviewer. They open the top pages and decide to refresh, expand, protect, or just monitor. The queue only decides the *order of review*.

Unit and output:one page per row. The output is a ranked list with a score and plain-language reason codes (e.g. "stale + visible", "declining with demand").

Cost of a wrong call:
- *False positive: reviewer time wasted on a healthy page.
- *False negative: a page with real search demand keeps losing visibility unnoticed.
- Because review capacity is limited (about 20–50 pages), I will judge it with Precision@20 / Precision@50.

Why data/ML can help:** a hand rule uses a few signals I picked by intuition. A model can weigh many measured signals at once. It only earns its place if it beats a transparent baseline on clients it has not seen. If it doesn't, that is still a result.

Why this is not just "train a model":** the real work is defining "declining" (size of drop, window, minimum volume), keeping outcome-derived fields like `trend_pct` out of the features, validating on held-out clients, and turning scores into actions a human can use.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import os, subprocess
import pandas as pd

# move to repo root (Colab or local)
if "google.colab" in str(get_ipython()):
    if not os.path.isdir("data/raw"):
        if not os.path.isdir("flyrank-ml-internship-starter"):
            subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/flyrank-bih/flyrank-ml-internship-starter"], check=True)
        os.chdir("flyrank-ml-internship-starter")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

# Number 1: size and prevalence of the problem
print(f"Pages: {len(df):,} across {df['client_id'].nunique()} clients")
print(f"Share labelled 'down': {df['is_declining'].mean():.1%}")

# Number 2: decline among pages that matter (visible pages)
visible = df[df["impressions_90d"] >= 500]
print(f"\nPages with >=500 impressions: {len(visible):,} ({len(visible)/len(df):.1%})")
print(f"Declining rate among those:   {visible['is_declining'].mean():.1%}")

# Number 3: spread across clients (why I plan a client-holdout validation)
per_client = df.groupby("client_id")["is_declining"].mean()
print(f"\nDecline rate per client: min {per_client.min():.1%}, "
      f"median {per_client.median():.1%}, max {per_client.max():.1%}")

Pages: 30,000 across 32 clients
Share labelled 'down': 54.2%

Pages with >=500 impressions: 16,726 (55.8%)
Declining rate among those:   59.6%

Decline rate per client: min 0.0%, median 52.4%, max 93.7%


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

I can say (observed / directional / decision-support):
- In this anonymized sample, many pages show a >20% impression drop between two 30-day windows.
- Some measured signals (e.g. freshness, visibility) are associated with that label here.
- A ranked list can help a reviewer decide which pages to look at first.

I can't say:**
- That a refresh *causes* recovery. This data has no experiment.
- That I found how Google's algorithm works.
- That "down" is a true future decline. It is a proxy label from the current window and could be noise, seasonality or consolidation.
- That results on this 30k slice carry over to the full warehouse.

Safety:only anonymized, aggregated data is used, with no client names, URLs or raw queries.

## Self-check

Before you submit, confirm each line honestly:

- [yes] Every section above is filled — markdown thinking AND the code that backs it
- [yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [yes] No client names, URLs, or private queries anywhere
- [yes] My claims use careful words: observed, measured, directional, decision-support
- [yes] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.